In [1]:
from glob import glob
from time import sleep
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as pf

from faster_whisper import WhisperModel

/home/grc/arep/wav/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Preparando o Dataset

In [2]:
spark = (
    SparkSession.builder
    .getOrCreate()
)

wav_files = glob("../audio/*.wav")
df = spark.createDataFrame([(f,) for f in wav_files], ["file_path"])
df = df.coalesce(4)

26/03/18 10:35:44 WARN Utils: Your hostname, grc resolves to a loopback address: 127.0.1.1; using 192.168.0.11 instead (on interface eno1)
26/03/18 10:35:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/18 10:35:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Criando a Função

O modelo precisa ser global, para evitar a leitura do disco novamente dele em cada aplicação da UDF. 

In [3]:
model = None

In [4]:
def get_model():
    from faster_whisper import WhisperModel
    global model
    
    if model is None:
        model = WhisperModel(
            "../model/",
            device="cpu",
            compute_type="int8"
        )

    return model

def transcribe_audio(file_path):
    model = get_model()

    segments, _ = model.transcribe(
        file_path,
        language="pt",
        condition_on_previous_text=False,
    )

    full_text = " ".join([segment.text for segment in segments])

    return full_text

In [5]:
def process_partition(iterator):
    for row in iterator:
        file_path = row.file_path
        
        try:
            transcription = transcribe_audio(file_path)
            dt = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            yield (file_path, transcription, dt)

        except Exception as e:
            dt = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            yield (file_path, f"ERRO: {str(e)}", dt)

26/03/18 10:35:58 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Para fazer essa transcrição, esta me consumindo 20gb de memória. Mas, a memória fica estável, não tendo picos.

In [6]:
df = spark.createDataFrame(
    df.rdd.mapPartitions(process_partition),
    ["file_path", "transcription", "datetime"]
)

In [7]:
df.select("file_path", "datetime", "transcription").orderBy("datetime").show(truncate=False)

+--------------------+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------